In [22]:
%run init_client.py

In [71]:
def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})

def chat_with_model(messages, system_message=None, temperature=1.0, stop=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
    }
    if system_message:
        params["system"] = system_message

    if stop:
        params["extra_body"]["stop"] = stop

    message = client.messages.create(**params)
    return message.content[0].text

In [24]:
messages = []

add_user_message(messages, "Define quantum computing in one sentence")

answer = chat_with_model(messages)
add_assistant_message(messages, answer)

print(messages)

[{'role': 'user', 'content': 'Define quantum computing in one sentence'}, {'role': 'assistant', 'content': '**Quantum computing** is a form of computation that harnesses quantum mechanical phenomena — such as superposition and entanglement — to process information in ways that can solve certain complex problems exponentially faster than classical computers.'}]


# Simple user chat interaction until user *exit*

In [51]:
messagesList = []
while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break
    add_user_message(messagesList, user_input)
    answer = chat_with_model(messagesList)
    add_assistant_message(messagesList, answer)
    print(f"Assistant: {answer}")

In [52]:
messages = []
system_message = """
You are a patient math tutor. 
You will help the user understand math concepts and solve problems step by step.
Do not provide the final answer immediately. Instead, guide the user through the problem-solving process, asking questions and providing hints as needed.
"""

add_user_message(messages, "Can you help me solve the equation 2x + 3 = 7?")
answer = chat_with_model(messages, system_message=system_message)
add_assistant_message(messages, answer)
messages


[{'role': 'user', 'content': 'Can you help me solve the equation 2x + 3 = 7?'},
 {'role': 'assistant',
  'content': "I'd be happy to help you solve that equation! Let's work through it together step by step.\n\nWe have: **2x + 3 = 7**\n\nThe goal is to **isolate x** (get x by itself on one side of the equation).\n\nLet me ask you this first:\n\n**What do you think we should do first to start isolating x?**\n\nHere's a hint: Look at the left side of the equation. There's a **+3** hanging out with the x term. How could we get rid of it? 🤔"}]

In [53]:
messages = []
add_user_message(messages, "Can you suggest top 3 action movies names of all time?")
answer = chat_with_model(messages, system_message=None, temperature=0.0)
add_assistant_message(messages, answer)
messages

KeyboardInterrupt: 

## stream messages

In [60]:
messages = []
add_user_message(messages, "Can you suggest best action movies names of all time?")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_bdrk_4kymevp7ytiqh7dfmkrqhdnatx6n6fhkczaujol2br3xgdy62aca', container=None, content=[], diagnostics=None, model='anthropic/claude-sonnet-4.6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo=None, input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard', speed='standard'), provider='Amazon Bedrock'), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=[], text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Best Action Movies', type='text_delta'), index=0, type='content_block_delta')
RawCo

In [61]:
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")


# Best Action Movies of All Time

Here are some highly acclaimed action movies across different eras:

## Classic Action
- **Die Hard** (1988)
- **Terminator 2: Judgment Day** (1991)
- **Predator** (1987)
- **RoboCop** (1987)

## Martial Arts / Fight Films
- **Enter the Dragon** (1973)
- **The Raid** (2011)
- **Ip Man** (2008)
- **Kill Bill** (2003)

## Spy / Thriller Action
- **Goldfinger** (1964)
- **Mission: Impossible - Fallout** (2018)
- **The Bourne Identity** (2002)

## Modern Action
- **Mad Max: Fury Road** (2015)
- **John Wick** (2014)
- **Gladiator** (2000)
- **The Dark Knight** (2008)

## Action Adventures
- **Raiders of the Lost Ark** (1981)
- **The Matrix** (1999)
- **Avengers: Endgame** (2019)
- **Top Gun: Maverick** (2022)

## Hollywood Blockbusters
- **Speed** (1994)
- **Lethal Weapon** (1987)
- **Heat** (1995)

---

Would you like recommendations based on a **specific genre, actor, or era?** 🎬

In [62]:
final_message = stream.get_final_message()
print(final_message)


ParsedMessage(id='msg_bdrk_2pi3rcxbjvc4x7tiivwf7yyiltnkerls6jid7o5volkogmjnn4la', container=None, content=[ParsedTextBlock(citations=[], text='# Best Action Movies of All Time\n\nHere are some highly acclaimed action movies across different eras:\n\n## Classic Action\n- **Die Hard** (1988)\n- **Terminator 2: Judgment Day** (1991)\n- **Predator** (1987)\n- **RoboCop** (1987)\n\n## Martial Arts / Fight Films\n- **Enter the Dragon** (1973)\n- **The Raid** (2011)\n- **Ip Man** (2008)\n- **Kill Bill** (2003)\n\n## Spy / Thriller Action\n- **Goldfinger** (1964)\n- **Mission: Impossible - Fallout** (2018)\n- **The Bourne Identity** (2002)\n\n## Modern Action\n- **Mad Max: Fury Road** (2015)\n- **John Wick** (2014)\n- **Gladiator** (2000)\n- **The Dark Knight** (2008)\n\n## Action Adventures\n- **Raiders of the Lost Ark** (1981)\n- **The Matrix** (1999)\n- **Avengers: Endgame** (2019)\n- **Top Gun: Maverick** (2022)\n\n## Hollywood Blockbusters\n- **Speed** (1994)\n- **Lethal Weapon** (1987)\n

# JSON format

In [74]:
messages = []
add_assistant_message(messages, "```json")
add_user_message(
    messages,
    "Generate an array of users with fields name, email, and age. Return only a JSON array."
)

answer = chat_with_model(messages, temperature=0.0, stop=["```"])
answer

'```json\n[\n  {\n    "name": "Alice Johnson",\n    "email": "alice.johnson@email.com",\n    "age": 28\n  },\n  {\n    "name": "Bob Smith",\n    "email": "bob.smith@email.com",\n    "age": 34\n  },\n  {\n    "name": "Carol White",\n    "email": "carol.white@email.com",\n    "age": 22\n  },\n  {\n    "name": "David Brown",\n    "email": "david.brown@email.com",\n    "age": 45\n  },\n  {\n    "name": "Eva Martinez",\n    "email": "eva.martinez@email.com",\n    "age": 31\n  },\n  {\n    "name": "Frank Wilson",\n    "email": "frank.wilson@email.com",\n    "age": 27\n  },\n  {\n    "name": "Grace Lee",\n    "email": "grace.lee@email.com",\n    "age": 39\n  },\n  {\n    "name": "Henry Taylor",\n    "email": "henry.taylor@email.com",\n    "age": 52\n  },\n  {\n    "name": "Isla Anderson",\n    "email": "isla.anderson@email.com",\n    "age": 24\n  },\n  {\n    "name": "Jack Thomas",\n    "email": "jack.thomas@email.com",\n    "age": 36\n  }\n]\n```'

In [76]:
import json
json_text = answer.strip().removeprefix("```json").removesuffix("```").strip()
json_array = json.loads(json_text)
json_array

[{'name': 'Alice Johnson', 'email': 'alice.johnson@email.com', 'age': 28},
 {'name': 'Bob Smith', 'email': 'bob.smith@email.com', 'age': 34},
 {'name': 'Carol White', 'email': 'carol.white@email.com', 'age': 22},
 {'name': 'David Brown', 'email': 'david.brown@email.com', 'age': 45},
 {'name': 'Eva Martinez', 'email': 'eva.martinez@email.com', 'age': 31},
 {'name': 'Frank Wilson', 'email': 'frank.wilson@email.com', 'age': 27},
 {'name': 'Grace Lee', 'email': 'grace.lee@email.com', 'age': 39},
 {'name': 'Henry Taylor', 'email': 'henry.taylor@email.com', 'age': 52},
 {'name': 'Isla Anderson', 'email': 'isla.anderson@email.com', 'age': 24},
 {'name': 'Jack Thomas', 'email': 'jack.thomas@email.com', 'age': 36}]

## JSON exercise

In [86]:
from IPython.display import Markdown
messages = []
add_assistant_message(messages, "Here are three different AWS cli commands in a single block without any comments:\n```bash")
add_user_message(messages, "Generate three different AWS cli commands. Each should be very short.")
answer = chat_with_model(messages, stop=["```"])
print(answer.strip())
Markdown(answer)

```bash
aws s3 ls
aws ec2 describe-instances
aws iam list-users
```


```bash
aws s3 ls
aws ec2 describe-instances
aws iam list-users
```